# Word2Vec ConShift results

Loads outputs from `code/anchors_conshift.py --method word2vec` (or legacy `anchors_word2vec_conshift.py`) in `results/anchors_word2vec/`:

- **`*_meta.txt`** — run settings and vocabulary sizes  
- **`*_shifts.tsv`** — each shared word, shift score, and **training token counts** in source/target corpus (`count_<corpus>` columns when you re-run the script)  
- **`*_anchors.txt`** — anchor words (lowest-shift quantile)  
- **`anchors_triple_intersection.txt`** — optional; anchors common to all three pairs  

**Top-N tables** are **one table per pair** with columns `word`, `shift`, `count_a`, `count_b` (source = a, target = b). Counts come from the TSV if present, else from `embeddings/word2vec/*.model`.

**Working directory:** the first code cell resolves the repo root and `results/anchors_word2vec`.

In [1]:
from pathlib import Path
import pandas as pd

TOP_N = 10
CORPORA = ["arxiv", "yelp", "ciao", "reddit"]

candidates_res = [
    Path("results/anchors_word2vec"),
    Path("../results/anchors_word2vec"),
    Path("../../results/anchors_word2vec"),
]
RESULTS_DIR = next((p.resolve() for p in candidates_res if p.is_dir()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError(
        "Could not find results/anchors_word2vec. "
        "cd to repo root or notebooks/ and re-run."
    )

candidates_repo = [RESULTS_DIR.parent, RESULTS_DIR.parent.parent]
REPO_ROOT = next(
    (p for p in candidates_repo if (p / "embeddings" / "word2vec").is_dir()),
    RESULTS_DIR.parent,
)
print(f"RESULTS_DIR = {RESULTS_DIR}")
print(f"REPO_ROOT   = {REPO_ROOT}")

RESULTS_DIR = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation\results\anchors_word2vec
REPO_ROOT   = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation


In [2]:
def load_meta(meta_path: Path) -> dict:
    out = {}
    for line in meta_path.read_text(encoding="utf-8").splitlines():
        if "\t" in line:
            k, v = line.split("\t", 1)
            out[k] = v
    out["pair"] = meta_path.name.replace("_meta.txt", "")
    return out


def parse_pair(pair_name: str):
    a, b = pair_name.split("_to_", 1)
    return a, b


_wv_cache: dict = {}


def wv_word_count(wv, w: str) -> int:
    if hasattr(wv, "get_vecattr"):
        try:
            c = wv.get_vecattr(w, "count")
            if c is not None:
                return int(c)
        except (KeyError, ValueError, TypeError):
            pass
    vocab = getattr(wv, "vocab", None)
    if vocab is not None and w in vocab:
        return int(vocab[w].count)
    return 0


def get_wv(corpus: str):
    if corpus not in _wv_cache:
        from gensim.models import Word2Vec

        path = REPO_ROOT / "embeddings" / "word2vec" / f"{corpus}.model"
        if not path.is_file():
            raise FileNotFoundError(f"Missing Word2Vec model: {path}")
        m = Word2Vec.load(str(path))
        _wv_cache[corpus] = m.wv
    return _wv_cache[corpus]


def attach_pair_counts(df: pd.DataFrame, src: str, tgt: str) -> pd.DataFrame:
    cs, ct = f"count_{src}", f"count_{tgt}"
    if cs in df.columns and ct in df.columns:
        return df
    wv_a, wv_b = get_wv(src), get_wv(tgt)
    out = df.copy()
    if cs not in out.columns:
        out[cs] = [wv_word_count(wv_a, w) for w in out["word"]]
    if ct not in out.columns:
        out[ct] = [wv_word_count(wv_b, w) for w in out["word"]]
    return out


def load_shifts(tsv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(tsv_path, sep="\t")
    pair_name = tsv_path.name.replace("_shifts.tsv", "")
    df["pair"] = pair_name
    src, tgt = parse_pair(pair_name)
    return attach_pair_counts(df, src, tgt)


def count_columns_for_pair(pair_name: str) -> list[str]:
    src, tgt = parse_pair(pair_name)
    return [f"count_{src}", f"count_{tgt}"]


def pairwise_top_n(
    df: pd.DataFrame, pair_name: str, top_n: int, *, largest: bool
) -> pd.DataFrame:
    """Columns: rank, word, shift, count_a, count_b (a=source, b=target)."""
    src, tgt = parse_pair(pair_name)
    ca, cb = f"count_{src}", f"count_{tgt}"
    part = (
        df.nlargest(top_n, "shift")
        if largest
        else df.nsmallest(top_n, "shift")
    )
    out = part[["word", "shift", ca, cb]].copy()
    out.insert(0, "rank", range(1, top_n + 1))
    return out.rename(columns={ca: "count_a", cb: "count_b"})


shift_files = sorted(RESULTS_DIR.glob("*_shifts.tsv"))
meta_files = sorted(RESULTS_DIR.glob("*_meta.txt"))
anchor_files = sorted(RESULTS_DIR.glob("*_anchors.txt"))
print("Shift files:", [p.name for p in shift_files])
print("Meta files:", [p.name for p in meta_files])

Shift files: ['arxiv_to_ciao_shifts.tsv', 'arxiv_to_yelp_shifts.tsv', 'yelp_to_ciao_shifts.tsv']
Meta files: ['arxiv_to_ciao_meta.txt', 'arxiv_to_yelp_meta.txt', 'yelp_to_ciao_meta.txt']


## Pairwise metadata

In [3]:
meta_df = pd.DataFrame([load_meta(p) for p in meta_files])
if not meta_df.empty:
    display(meta_df.set_index("pair"))

,source,target,shared_vocab,anchor_count,quantile,iterations,stopwords_removed,min_rank_per_corpus,min_word_length
pair,,,,,,,,,
arxiv_to_ciao,arxiv,ciao,8385,1258,0.15,9,True,None,4
arxiv_to_yelp,arxiv,yelp,7698,1155,0.15,10,True,None,4
yelp_to_ciao,yelp,ciao,31768,4766,0.15,11,True,None,4


## Top *N* most shifted (highest shift) per pair

One table **per pair** (`arxiv_to_ciao`, `arxiv_to_yelp`, `yelp_to_ciao`). Columns: **`word`**, **`shift`**, **`count_a`**, **`count_b`**, where **a** = source corpus and **b** = target corpus in the pair name. Set `TOP_N` in the first cell.

In [4]:
from IPython.display import Markdown, display

for p in shift_files:
    df = load_shifts(p)
    pair_name = df["pair"].iloc[0]
    src, tgt = parse_pair(pair_name)
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=True))

### `arxiv_to_ciao` — **count_a** = `arxiv` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
1235,1,cloud,1.342378,447,356
8066,2,valued,1.307182,319,181
2783,3,exhibits,1.295806,108,365
2117,4,determined,1.287903,339,1316
3472,5,hardness,1.286723,205,58
7349,6,supermarket,1.254374,26,2041
4612,7,membership,1.251068,130,379
2869,8,extensions,1.249576,249,89
6180,9,relaxation,1.233610,204,206
5804,10,proof,1.230939,939,1115


### `arxiv_to_yelp` — **count_a** = `arxiv` (source), **count_b** = `yelp` (target)

,rank,word,shift,count_a,count_b
7401,1,valued,1.324418,319,168
4339,2,minor,1.279824,116,1482
367,3,apps,1.248800,8,2016
2892,4,food,1.243678,39,217235
4256,5,membership,1.241707,130,1385
776,6,bound,1.240849,2284,411
989,7,chain,1.236250,396,6791
2996,8,functions,1.235979,1766,133
4564,9,norm,1.229030,378,588
4067,10,loop,1.226324,206,867


### `yelp_to_ciao` — **count_a** = `yelp` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
20306,1,pegs,1.251361,72,248
24547,2,scramble,1.198027,481,66
15448,3,killer,1.175712,1401,2077
12951,4,haven,1.165392,390,5969
17957,5,mode,1.159711,349,3063
10707,6,flare,1.150760,240,161
5268,7,cobb,1.130807,403,124
13645,8,hubs,1.126845,519,27
6208,9,cordial,1.119995,259,125
26522,10,spotty,1.098679,616,146


## Top *N* most stable (lowest shift) per pair

Same as above: **three tables**, columns `word`, `shift`, `count_a`, `count_b`.

In [5]:
from IPython.display import Markdown, display

for p in shift_files:
    df = load_shifts(p)
    pair_name = df["pair"].iloc[0]
    src, tgt = parse_pair(pair_name)
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=False))

### `arxiv_to_ciao` — **count_a** = `arxiv` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
6768,1,sharma,0.075043,6,14
4723,2,misuse,0.077969,17,22
7812,3,triplet,0.087102,10,5
614,4,avalanches,0.089509,5,6
2144,5,diagnosing,0.090759,5,8
4396,6,locomotion,0.095074,11,9
8378,7,zhou,0.096782,7,7
4973,8,noun,0.098731,7,18
3559,9,hitherto,0.099494,15,15
1135,10,chong,0.105395,6,8


### `arxiv_to_yelp` — **count_a** = `arxiv` (source), **count_b** = `yelp` (target)

,rank,word,shift,count_a,count_b
3001,1,funding,0.083282,17,53
4589,2,noun,0.083875,7,20
4362,3,misuse,0.104144,17,7
4778,4,overlapped,0.104629,9,8
714,5,biopsy,0.107699,9,13
5645,6,registries,0.107786,10,18
2003,7,dialect,0.112252,9,6
125,8,administrative,0.112706,18,32
227,9,aligning,0.112917,11,6
1699,10,crimes,0.113418,5,17


### `yelp_to_ciao` — **count_a** = `yelp` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
26094,1,solidarity,0.031594,7,13
18311,2,muhammad,0.031916,5,32
26719,3,stances,0.033503,7,18
4149,4,carpal,0.035267,7,10
8713,5,ducking,0.035482,20,17
13626,6,howell,0.036049,13,22
31237,7,wily,0.036794,7,19
10368,8,felipe,0.037337,11,12
22941,9,reinstated,0.037904,10,12
11307,10,friedman,0.037980,6,20


## Words in all loaded pairwise vocabularies (intersection)

Merge shift scores on `word` and summarize **mean shift** across the three pairs. Tables include **counts** `count_arxiv`, `count_yelp`, `count_ciao` (from Word2Vec training stats).

In [6]:
if len(shift_files) >= 2:
    merged = None
    for p in shift_files:
        df = pd.read_csv(p, sep="\t")
        pair = p.name.replace("_shifts.tsv", "")
        sub = df.rename(columns={"shift": f"shift_{pair}"})[["word", f"shift_{pair}"]]
        merged = sub if merged is None else merged.merge(sub, on="word", how="inner")
    shift_cols = [c for c in merged.columns if c.startswith("shift_")]
    merged["mean_shift"] = merged[shift_cols].mean(axis=1)
    for c in CORPORA:
        col = f"count_{c}"
        if col not in merged.columns:
            wv = get_wv(c)
            merged[col] = [wv_word_count(wv, w) for w in merged["word"]]
    count_cols = [f"count_{c}" for c in CORPORA]
    print(f"Words in all {len(shift_files)} pairwise vocabularies: {len(merged)}")
    show_cols = ["word", "mean_shift"] + shift_cols + count_cols
    display(merged.nlargest(TOP_N, "mean_shift")[show_cols])
    display(merged.nsmallest(TOP_N, "mean_shift")[show_cols])
else:
    print("Need at least two shift files for merge.")

Words in all 3 pairwise vocabularies: 7441


,word,mean_shift,shift_arxiv_to_ciao,shift_arxiv_to_yelp,shift_yelp_to_ciao,count_arxiv,count_yelp,count_ciao
7156,valued,1.040510,1.307182,1.324418,0.489930,319,168,181
3671,joint,1.014453,0.753096,1.193394,1.096870,484,6534,477
5379,recommendation,1.010703,1.077005,1.123027,0.832078,103,2833,1453
3906,local,1.009252,1.130216,1.181146,0.716395,1405,19422,8761
3673,joints,1.003988,0.895995,1.022681,1.093288,10,1780,299
601,basis,0.982685,1.198090,1.151534,0.598432,561,1327,2416
6301,spot,0.972353,1.030488,1.050914,0.835656,25,19879,3176
5150,proof,0.963257,1.230939,1.145504,0.513329,939,448,1115
4213,misses,0.961362,0.829011,1.014530,1.040545,11,307,267
945,chain,0.960833,0.998501,1.236250,0.647747,396,6791,1144


,word,mean_shift,shift_arxiv_to_ciao,shift_arxiv_to_yelp,shift_yelp_to_ciao,count_arxiv,count_yelp,count_ciao
4436,noun,0.080264,0.098731,0.083875,0.058186,7,20,18
4218,misuse,0.081032,0.077969,0.104144,0.060984,17,7,22
4002,maneuvers,0.109440,0.105872,0.144191,0.078258,16,9,8
2872,friedman,0.112582,0.160693,0.139073,0.037980,8,6,20
158,advocates,0.112682,0.128227,0.125753,0.084067,5,13,20
220,aligning,0.114759,0.135804,0.112917,0.095555,11,6,13
3695,kahn,0.116641,0.120086,0.148141,0.081696,9,14,20
7107,unveiling,0.123328,0.106811,0.163546,0.099627,6,10,15
5925,segregation,0.123687,0.124457,0.117039,0.129565,8,11,43
177,agencies,0.124453,0.148167,0.115539,0.109654,32,44,78


## Anchor list sizes

In [7]:
anchor_counts = []
for p in anchor_files:
    n = sum(1 for _ in p.open(encoding="utf-8"))
    anchor_counts.append({"pair": p.name.replace("_anchors.txt", ""), "anchor_lines": n})
display(pd.DataFrame(anchor_counts))

,pair,anchor_lines
0,arxiv_to_ciao,1258
1,arxiv_to_yelp,1155
2,yelp_to_ciao,4766


## Triple intersection anchors (if generated)

Created with `python code/anchors_conshift.py --method word2vec --mode triple-intersection`.

In [8]:
triple_path = RESULTS_DIR / "anchors_triple_intersection.txt"
if triple_path.is_file():
    words = [ln.strip() for ln in triple_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    print(f"Triple intersection count: {len(words)}")
    display(pd.DataFrame({"word": words[:200]}))  # preview
else:
    print(f"No file at {triple_path}")

No file at C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation\results\anchors_word2vec\anchors_triple_intersection.txt


## Full shift tables (optional)

Uncomment to load every pairwise file into one long dataframe (can be large).

In [9]:
# all_shifts = pd.concat([load_shifts(p) for p in shift_files], ignore_index=True)
# display(all_shifts.groupby("pair")["shift"].describe())